In [ ]:
# Colab-only fallback; safe to skip locally if NumPy/Matplotlib are already installed.
!pip install -q "numpy>=1.24" "matplotlib>=3.8"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FranQuant/the-ai-engineer/blob/main/capstones/week01_gd_optimization/gd_capstone.ipynb)

# Gradient Descent and SGD on a Piecewise Non-Smooth Objective — Week 1 Capstone

This notebook implements the Week 1 gradient-descent capstone from *The AI Engineer* program.

We study gradient descent and stochastic gradient descent on

$$
f(x)=\left|\tfrac{1}{2}x^3-\tfrac{3}{2}x^2\right|+\tfrac{1}{2}x,
$$

a 1-D objective with a kink at $x=3$ and a unique global minimizer at

$$
x^\star = 1 - \tfrac{2\sqrt{3}}{3} \approx -0.1547.
$$

A quadratic baseline $q(x)=\tfrac{1}{2}x^2$ is included as a simple stability reference for step-size behavior.

The notebook is fully self-contained: all data are generated programmatically, runs are seeded, and figures are saved deterministically.

## 1. Setup & Hyperparameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_STAR = 1 - (2 / 3) * np.sqrt(3)
EPS = 1e-6
TOL = 1e-4
T_MAX = 5000

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

print(f"X_STAR = {X_STAR:.6f}")

### Core configuration

| Parameter | Value | Purpose |
|---|---:|---|
| SGD seeds | `101, 202, 303` + `100–119` | Explicit per-run `default_rng` seeds (display + aggregation) |
| `X_STAR` | `1 - 2√3 / 3 ≈ -0.1547` | Analytic global minimizer |
| `EPS` | `1e-6` | Finite-difference step for gradient checks |
| `TOL` | `1e-4` | Objective-gap tolerance |
| `T_MAX` | `5000` | Maximum iteration budget |
| GD step sizes | `0.05, 0.10, 0.15, 0.20` | Sweep for deterministic GD |
| SGD setup | fixed below | Constant and diminishing schedules |

## 2. Objective Function & Derivative

Define $g(x) = \tfrac{1}{2}x^3 - \tfrac{3}{2}x^2 = \tfrac{x^2(x-3)}{2}$. The objective is

$$f(x) = |g(x)| + \tfrac{1}{2}x.$$

**Why the kink at $x = 3$.**  
The absolute value creates a kink wherever $g(x) = 0$ *and* $g'(x) \neq 0$.
At $x = 3$: $g(3) = 0$ and $g'(3) = \tfrac{3}{2}(9) - 3(3) = 4.5 \neq 0$, so $x = 3$ is a genuine kink.
At $x = 0$: $g(0) = 0$ and $g'(0) = 0$, so $f$ is smooth at $x = 0$.

**Piecewise derivative** (valid off the kink):

$$f'(x) = \operatorname{sign}(g(x))\cdot g'(x) + \tfrac{1}{2}, \qquad g'(x) = \tfrac{3}{2}x^2 - 3x.$$

**Global minimizer.**  
For $x < 3$, we have $g(x) \le 0$, with equality only at $x = 0$. On this branch, $f'(x) = -g'(x) + \tfrac{1}{2} = -\tfrac{3}{2}x^2 + 3x + \tfrac{1}{2}$.
Setting $f'(x) = 0$ and solving the quadratic gives $x^\star = 1 - \tfrac{2\sqrt{3}}{3} \approx -0.1547$.

In [ ]:
def g(x):
    return 0.5*x**3 - 1.5*x**2

def f(x):
    return np.abs(g(x)) + 0.5*x

def df(x):
    """Derivative of f; exact off the kink.

    The lone non-differentiable point is x=3, where the subdifferential is the
    interval [-4, 5]. Branching on ``x >= 3`` is exact: it avoids the
    floating-point cancellation in g(x) one ulp below the kink and selects
    the right-derivative (+5) at x = 3.
    """
    dg   = 1.5*x**2 - 3.0*x          # g'(x)
    sign = np.where(x >= 3.0, 1.0, -1.0)   # sign(g) = +1 iff x >= 3, since g = x^2 (x-3)/2
    return sign * dg + 0.5

def f_gap(x):
    return f(x) - f(X_STAR)

# Float-exactness at the kink's neighbors (catches cancellation misbranching):
assert df(np.nextafter(3.0, -np.inf)) < 0 and df(np.nextafter(3.0, np.inf)) > 0
print("df branch check at nextafter(3): PASS")

## 3. Function Landscape

In [ ]:
xs_plot = np.linspace(-1, 4.5, 600)
left_xs = xs_plot[xs_plot < 3.0]
right_xs = xs_plot[xs_plot > 3.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# --- f(x) ---
ax1.plot(xs_plot, f(xs_plot), linewidth=2, label='f(x)')
ax1.axvline(3.0,    linestyle='--', color='grey',      linewidth=1.2, label='x = 3  (kink)')
ax1.axvline(X_STAR, linestyle='--', color='steelblue', linewidth=1.2,
            label=f'x* \u2248 {X_STAR:.4f}  (global min)')
ax1.scatter([X_STAR], [f(X_STAR)], color='steelblue', zorder=5, s=60)
ax1.set_title('Objective  f(x) = |g(x)| + 0.5x')
ax1.set_xlabel('x')
ax1.set_ylabel('f(x)')
ax1.legend(fontsize=9)

# --- f'(x) ---
ax2.plot(left_xs, df(left_xs), linewidth=2, color='darkorange', label="f'(x)")
ax2.plot(right_xs, df(right_xs), linewidth=2, color='darkorange', label='_nolegend_')
ax2.plot(3.0, -4.0, marker='o', markersize=7, markerfacecolor='white',
         markeredgecolor='darkorange', markeredgewidth=1.5, linestyle='None',
         label='one-sided limits at x = 3')
ax2.plot(3.0, 5.0, marker='o', markersize=7, markerfacecolor='white',
         markeredgecolor='darkorange', markeredgewidth=1.5, linestyle='None',
         label='_nolegend_')
ax2.axvline(3.0, linestyle='--', color='grey', linewidth=1.2, label='x = 3  (kink)')
ax2.axhline(0,   color='black', linewidth=0.8)
ax2.set_title("Derivative  f'(x)  (piecewise)")
ax2.set_xlabel('x')
ax2.set_ylabel("f'(x)")
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_01_landscape.png', dpi=150, bbox_inches='tight')
plt.show()

**Observations.** At the kink the one-sided slopes are $\lim_{x \to 3^-} f'(x) = -4$ and $\lim_{x \to 3^+} f'(x) = 5$, so $\partial f(3) = [-4,\, 5]$: no single gradient exists there. The implementation resolves this with the right-branch convention documented in `df` (it returns $+5$ at $x = 3$).

## 4. Gradient Check

Before trusting `df` inside GD, compare it to a centered finite difference at each test point:

$$f'(x_0) \approx \frac{f(x_0 + \varepsilon) - f(x_0 - \varepsilon)}{2\varepsilon}, \qquad \varepsilon = 10^{-6}.$$

Test points cover both smooth branches and $x^\star$, and deliberately exclude the kink at $x = 3$, where no derivative exists for the check to approximate.

In [ ]:
test_points = [-0.8, X_STAR, 0.5, 1.0, 2.5, 3.5, 4.2]

print('=== Finite-difference gradient check ===')
print(f"{'x':>8}  {'analytic':>12}  {'finite-diff':>12}  {'abs error':>10}")
print('-' * 50)
errors = []
for x0 in test_points:
    fd  = (f(x0 + EPS) - f(x0 - EPS)) / (2 * EPS)
    ana = float(df(x0))
    err = abs(ana - fd)
    errors.append(err)
    print(f"{x0:>8.4f}  {ana:>12.6f}  {fd:>12.6f}  {err:>10.2e}")

max_err = max(errors)
print(f"\nMax absolute error across all test points: {max_err:.2e}")
assert max_err < 1e-6, f"Gradient check failed: {max_err:.2e}"

All errors are below $10^{-6}$ (enforced by the assert at runtime). At a kink, a centered difference straddles the jump in one-sided slopes and returns their average rather than a derivative — here $(5 + (-4))/2 = 0.5$ — which is why $x = 3$ is excluded.

## 5. Gradient Descent — Implementation

The GD update rule is

$$x_{t+1} = x_t - \eta\, f'(x_t).$$

We include two early-exit conditions:

1. **Convergence**: $f(x_t) - f(x^\star) < \text{TOL}$ — an *oracle* stopping rule, usable here because $x^\star$ is known analytically; unavailable in real problems.
2. **Divergence guard**: $|x_t| > 10^4$ — the iterate has left a recoverable region; we record `NaN` and stop.

At the one nondifferentiable point $x = 3$, `df` returns the right-hand subgradient $+5$ (see its docstring).

In [ ]:
def gd(x0, eta, T=T_MAX):
    xs = [x0]
    x  = x0
    for _ in range(T):
        x = x - eta * df(x)
        xs.append(x)
        if f_gap(x) < TOL:
            break
        if abs(x) > 1e4:        # divergence guard
            xs.append(np.nan)
            break
    return np.array(xs)

## 6. GD Trajectories — Multiple Initializations

In [ ]:
x0_conv = [-0.8, 0.0, 0.5, 1.5]
x0_osc  = [2.8, 3.5]
T_plot = 60
eta_traj = 0.05

steps = np.arange(T_plot + 1)
traj_style = dict(linewidth=1.8, marker='o', markersize=2.6, markevery=6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, x0_list, title, legend_loc in [
    (ax1, x0_conv, 'GD Trajectories — Convergent Initializations', 'upper right'),
    (ax2, x0_osc, 'GD Trajectories — Kink-Region / Oscillatory Initializations', 'upper left'),
]:
    for x0 in x0_list:
        xs = gd(x0, eta_traj, T=T_plot)
        ax.plot(steps[:len(xs)], xs, label=f'x0 = {x0}', **traj_style)

    ax.axhline(X_STAR, linestyle='--', color='black', linewidth=1.2,
               label=f'x* \u2248 {X_STAR:.4f}')
    ax.axhline(3.0, linestyle=':', color='grey', linewidth=1.0, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Iteration')
    ax.set_xlim(0, T_plot)
    ax.set_ylim(-1.0, 4.0)
    ax.legend(fontsize=8, loc=legend_loc)

ax1.set_ylabel('x_t')
ax2.tick_params(labelleft=False)

plt.tight_layout()
plt.savefig('fig_02_gd_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

# Verify the 5000-step claim made below: neither kink-region start converges.
for x0 in x0_osc:
    xs_long = gd(x0, eta_traj, T=5000)
    tail = xs_long[-500:]
    print(f"x0 = {x0}: converged = {bool(f_gap(xs_long[-1]) < TOL)}, "
          f"steps run = {len(xs_long)-1}, "
          f"late-stage range = [{np.nanmin(tail):.3f}, {np.nanmax(tail):.3f}]")

**Convergent vs. oscillatory starts.** For $\eta = 0.05$, starts $x_0 \in \{-0.8,\, 0.0,\, 0.5,\, 1.5\}$ converge to $x^\star$, while $x_0 \in \{2.8,\, 3.5\}$ enter persistent oscillation around the kink and do not converge within 5000 steps (demonstrated by the printed check above at runtime).

**Why.** On the $g < 0$ branch, $f'(x) = -\tfrac{3}{2}x^2 + 3x + \tfrac{1}{2}$ vanishes at $x^\star$ and at a local maximum $x_{\text{max}} = 1 + \tfrac{2\sqrt{3}}{3} \approx 2.155$. For $x \in (x_{\text{max}},\, 3)$, $f' < 0$ pushes iterates right; for $x > 3$, $f' \geq 5 > 0$ pushes them left. Since $0 \in \partial f(3) = [-4,\, 5]$, the kink is itself a *nonsmooth local minimizer* ($f(3) = 1.5$): fixed-step subgradient descent can approach it but never settle, because the subgradient jumps from $-4$ to $+5$ across $x = 3$. Smaller steps shrink the oscillation amplitude but do not produce convergence. For this experiment, $x_{\text{max}}$ separates the two behaviors.

## 7. Protocol Trajectories

The handout's protocol-trajectory figure (Fig. 4) uses the protocol $x_0 \in \{-1.0,\, 0.5,\, 2.0\}$ with $\eta = 0.15$. All three starts lie below $x_{\text{max}} \approx 2.155$, so they head toward $x^\star$. The figure below overlays the three iterates on $f(x)$.

In [ ]:
x0_protocol  = [-1.0, 0.5, 2.0]
eta_protocol = 0.15
T_protocol   = 200

proto_trajs = {x0: gd(x0, eta_protocol, T=T_protocol) for x0 in x0_protocol}

xs_bg = np.linspace(-1, 3.5, 600)
colors = ['tab:blue', 'tab:orange', 'tab:green']

fig, ax = plt.subplots(figsize=(9, 5))

# Background: f(x)
ax.plot(xs_bg, f(xs_bg), color='0.30', linewidth=2, label='f(x)', zorder=1)

# Protocol trajectories: mark start and end on the curve
for (x0, xs), color in zip(proto_trajs.items(), colors):
    fx = f(np.asarray(xs))
    ax.plot(xs, fx, 'o-', color=color, markersize=3, linewidth=1.4,
            label=f'x0 = {x0}', zorder=2, markevery=10)
    ax.scatter([xs[0]], [f(xs[0])], color=color, marker='s', s=60, zorder=4)
    ax.scatter([xs[-1]], [f(xs[-1])], color=color, marker='*', s=100, zorder=4)

# Reference lines
ax.axvline(3.0, linestyle='--', color='grey', linewidth=1.2, label='x = 3  (kink)')
ax.axhline(f(X_STAR), linestyle='--', color='black', linewidth=1.0,
           label=f'f(x*) ~ {f(X_STAR):.4f}')

ax.set_title(
    f'Protocol Trajectories - Handout Fig. 4 Replication\n'
    f'x0 in {{{x0_protocol[0]}, {x0_protocol[1]}, {x0_protocol[2]}}}, '
    f'eta = {eta_protocol}, K = {T_protocol}'
)
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.set_xlim(-1, 3.5)
ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.savefig('fig_03_protocol_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. GD Step-Size Sweep

In [ ]:
# GD step-size sweep over four step sizes
etas_sweep = [0.05, 0.10, 0.15, 0.20]
x0_sweep   = 2.0
T_sweep    = 200

trajectories = {eta: gd(x0_sweep, eta, T=T_sweep) for eta in etas_sweep}

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8))

# Top: x_t vs iteration
for eta in etas_sweep:
    xs = trajectories[eta]
    ax1.plot(xs, label=f'η = {eta}')
ax1.axhline(X_STAR, linestyle='--', color='black', linewidth=1,
            label=f'x* ≈ {X_STAR:.4f}')
ax1.set_title('GD Step-Size Sweep — x_t vs Iteration')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('x_t')
ax1.legend(fontsize=9)

# Bottom: f_gap on log scale
for eta in etas_sweep:
    xs = trajectories[eta]
    gaps = np.maximum(f_gap(np.asarray(xs)), 1e-12)
    ax2.plot(gaps, label=f'η = {eta}')
ax2.axhline(TOL, linestyle='--', color='black', linewidth=1, label=f'TOL = {TOL}')
ax2.set_yscale('log')
ax2.set_title('GD Step-Size Sweep — f_gap(x_t)  (log scale)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('f_gap(x_t)')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_04_step_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

**Step-size sensitivity.** For an $L$-Lipschitz gradient, GD with $\eta < 2/L$ descends on each smooth branch: locally $\eta \lesssim 0.58$ near $x^\star$ and $\eta \lesssim 0.33$ on the branches adjoining the kink. These bounds govern stability *within* a branch — the oscillation at $x = 3$ is caused by the subgradient sign jump, which no step size cures (see Section 6).

The sweep starts at $x_0 = 2.0 < x_{\text{max}}$, so all four step sizes converge, with speed increasing monotonically in $\eta$: slowest at $\eta = 0.05$, fastest at $\eta = 0.20$ (still within the local bound $0.20 < 0.33$). All four runs reach TOL well within 200 steps.

## 9. Local Step Geometry

A single gradient step is easiest to read when we show the tangent line and the update together. We pick a point in the smooth basin, draw the tangent, and mark the next iterate from one GD step.

In [ ]:
x0_geom = 2.0
eta_geom = 0.20
x1_geom = x0_geom - eta_geom * df(x0_geom)
f0_geom = f(x0_geom)
f1_geom = f(x1_geom)

xs_geom = np.linspace(x0_geom - 0.9, x0_geom + 0.9, 400)
tangent_geom = f0_geom + df(x0_geom) * (xs_geom - x0_geom)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(xs_geom, f(xs_geom), linewidth=2, label='f(x)')
ax.plot(xs_geom, tangent_geom, linestyle='--', color='crimson', linewidth=2,
        label='Local linear model at x0')
ax.scatter([x0_geom], [f0_geom], color='black', zorder=5, s=55, label=f'x0 = {x0_geom}')
ax.scatter([x1_geom], [f1_geom], color='steelblue', zorder=5, s=55,
           label=f'x1 = {x1_geom:.3f}')
ax.annotate('', xy=(x1_geom, f0_geom), xytext=(x0_geom, f0_geom),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.text((x0_geom + x1_geom) / 2, f0_geom + 0.12, r"$\Delta x = -\eta f'(x_0)$",
        ha='center', va='bottom', fontsize=10,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.8, pad=0.2))
ax.axvline(x0_geom, linestyle=':', color='0.75', linewidth=1, alpha=0.8)
ax.axvline(x1_geom, linestyle=':', color='0.75', linewidth=1, alpha=0.8)
ax.set_title('Local Step Geometry — one GD update from the tangent line')
ax.set_xlabel('x')
ax.set_ylabel('f(x)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_05_step_geometry.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Stochastic Gradient Descent (SGD)

We keep the objective deterministic and perturb only the gradient estimate:

$$\tilde{g}_t = f'(x_t) + \sigma \xi_t, \qquad \xi_t \sim \mathcal{N}(0,1).$$

The SGD update is $x_{t+1} = x_t - \eta\, \tilde{g}_t$. Fixed seeds make the sample paths reproducible, and a constant step size makes the noise floor easy to see.

> **Protocol note:** the SGD experiments deviate from the handout protocol (x0 in {-1.0, 0.5, 2.0}, K = 200) by starting at x0 = 1.8 with T = 300; the start adds margin below the basin boundary at x ~ 2.155 so the figures isolate the noise floor rather than basin-escape events, and the longer horizon makes the constant-step floor visually unambiguous.

In [ ]:
SGD_X0    = 1.8
SGD_SIGMA = 0.35
SGD_ETA   = 0.10
SGD_ETA0  = 0.10
SGD_GAMMA = 0.05
SGD_T     = 300
SGD_SEEDS = [101, 202, 303]

def sgd(x0, sigma, T, seed, eta_fn):
    rng = np.random.default_rng(seed)
    xs = [x0]
    x = x0
    for k in range(T):
        x = x - eta_fn(k) * (df(x) + sigma * rng.normal())
        xs.append(x)
        if abs(x) > 1e4:
            xs.append(np.nan)
            break
    return np.array(xs)

eta_const = lambda k: SGD_ETA
eta_dim   = lambda k: SGD_ETA0 / (1.0 + SGD_GAMMA * k)

fig6_seeds = range(100, 120)
fig6_paths = {seed: sgd(SGD_X0, SGD_SIGMA, SGD_T, seed, eta_const)
              for seed in dict.fromkeys([*SGD_SEEDS, *fig6_seeds])}
fig6_gaps = np.vstack([
    np.maximum(f_gap(np.asarray(fig6_paths[seed])), 1e-12)
    for seed in fig6_seeds
])
med_gap = np.median(fig6_gaps, axis=0)
q1_gap = np.percentile(fig6_gaps, 25, axis=0)
q3_gap = np.percentile(fig6_gaps, 75, axis=0)
steps = np.arange(SGD_T + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for seed in SGD_SEEDS:
    ax1.plot(fig6_paths[seed], label=f'seed = {seed}')

ax1.axhline(X_STAR, linestyle='--', color='black', linewidth=1.2,
           label=f'x* \u2248 {X_STAR:.4f}')
ax1.set_title('SGD Sample Paths — Constant Step Size')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('x_t')
ax1.set_xlim(0, SGD_T)
ax1.legend(fontsize=9)

ax2.plot(steps, med_gap, color='tab:blue', label=f'median gap across {len(fig6_seeds)} seeds')
ax2.fill_between(steps, q1_gap, q3_gap, color='tab:blue', alpha=0.18,
                 label='25th-75th percentile band')
ax2.axhline(TOL, linestyle='--', color='black', linewidth=1, label=f'TOL = {TOL}')
ax2.set_yscale('log')
ax2.set_title('SGD Objective Gap — Constant Step Size (Aggregated Across Seeds)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('f_gap(x_t)')
ax2.set_xlim(0, SGD_T)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_06_sgd_paths.png', dpi=150, bbox_inches='tight')
plt.show()

Figure 6: three seeded constant-step sample paths (left) and the median objective gap with a 25th–75th percentile band across 20 seeds (right). Constant-step SGD settles into a noise-dominated neighborhood of $x^\star$ rather than converging exactly.

## 11. Constant vs Diminishing Schedule

We compare a constant step size with the diminishing schedule defined below, using the same start point, noise scale, and 20 fixed seeds. The figure reports the median objective gap together with a 25th–75th percentile band.

$$\eta_k = \eta_0 / (1 + \gamma k).$$

In [ ]:
xs_const = sgd(SGD_X0, SGD_SIGMA, SGD_T, SGD_SEEDS[0], eta_const)
xs_dim = sgd(SGD_X0, SGD_SIGMA, SGD_T, SGD_SEEDS[0], eta_dim)
schedule_seeds = list(range(100, 120))

def gap_series(xs):
    return np.maximum(f_gap(np.asarray(xs)), 1e-12)

def schedule_gap_stats(eta_fn):
    gaps = np.vstack([
        gap_series(sgd(SGD_X0, SGD_SIGMA, SGD_T, seed, eta_fn))
        for seed in schedule_seeds
    ])
    return np.median(gaps, axis=0), np.percentile(gaps, 25, axis=0), np.percentile(gaps, 75, axis=0)

med_const, q1_const, q3_const = schedule_gap_stats(eta_const)
med_dim, q1_dim, q3_dim = schedule_gap_stats(eta_dim)
steps = np.arange(SGD_T + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, med_const, color='tab:blue', label=f'constant eta = {SGD_ETA}')
ax.fill_between(steps, q1_const, q3_const, color='tab:blue', alpha=0.18)
ax.plot(steps, med_dim, color='tab:orange', label=f'diminishing eta0 = {SGD_ETA0}, gamma = {SGD_GAMMA}')
ax.fill_between(steps, q1_dim, q3_dim, color='tab:orange', alpha=0.18)
ax.axhline(TOL, linestyle='--', color='black', linewidth=1, label=f'TOL = {TOL}')
ax.set_yscale('log')
ax.set_title('SGD Objective Gap — Constant vs Diminishing Schedule (Aggregated Across Seeds)')
ax.set_xlabel('Iteration')
ax.set_ylabel('f_gap(x_t)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('fig_07_sgd_schedule_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Metrics Summary

The table below uses the same start point `x0 = 1.8`. For SGD we keep the seed fixed at `101` so the comparison is reproducible.

In [ ]:
def summarize_run(xs, tol=TOL):
    gaps = np.maximum(f_gap(np.asarray(xs)), 0.0)
    hit = np.where(gaps < tol)[0]
    steps_to_tol = int(hit[0]) if len(hit) else None
    return gaps[-1], gaps.min(), steps_to_tol

gd_eta_010 = gd(SGD_X0, 0.10, T=SGD_T)

runs = [
    ('GD eta=0.10', gd_eta_010),
    ('SGD constant eta (seed=101)', xs_const),
    ('SGD diminishing eta_k (seed=101)', xs_dim),
]

print(f"{'method':<34} {'final_gap':>12} {'best_gap':>12} {'steps_to_tol':>12}")
print('-' * 74)
for method, xs in runs:
    final_gap, best_gap, steps_to_tol = summarize_run(xs)
    steps_text = '--' if steps_to_tol is None else f'{steps_to_tol:d}'
    print(f"{method:<34} {final_gap:>12.2e} {best_gap:>12.2e} {steps_text:>12}")

For SGD, `steps_to_tol` is the first iteration where the gap falls below tolerance; noisy paths can later move back above that threshold.

GD here descends monotonically and stops at the first sub-TOL iterate, so its final and best gaps coincide. For SGD, the constant schedule can reach a very low transient best gap but finish at a higher long-run noise floor, while the diminishing schedule ends with a smaller final gap.

## 13. Quadratic Baseline — Stability Reference

Quadratic baseline: $q(x) = \tfrac{1}{2}x^2$ is a clean reference case with gradient Lipschitz constant $L = 1$. The GD update is $x_{t+1} = (1 - \eta)x_t$, so the exact stability threshold is convergence for $0 < \eta < 2$ and divergence for $\eta > 2$.

In [ ]:
def q(x):
    return 0.5 * x**2

def dq(x):
    return x

def gd_quadratic(x0, eta, T=T_MAX):
    xs = [x0]
    x  = x0
    for _ in range(T):
        x = x - eta * dq(x)
        if abs(q(x) - q(0.0)) < TOL:
            xs.append(x)   # record the converged point before stopping
            break
        if abs(x) > 1e6:
            xs.append(np.nan)
            break
        xs.append(x)
    return np.array(xs)

x0_quad      = 4.0
stable_etas  = [0.05, 0.10, 0.15, 0.20]
unstable_eta = 2.1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 8))

# Top: stable step sizes
for eta in stable_etas:
    xs = gd_quadratic(x0_quad, eta)
    ax1.plot(xs, label=f'\u03b7 = {eta}')
ax1.axhline(0.0, linestyle='--', color='black', linewidth=1, label='x* = 0')
ax1.set_title('Quadratic GD — Stable Step Sizes  (0 < \u03b7 < 2)')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('x_t')
ax1.legend()

# Bottom: divergent step size
xs_div = gd_quadratic(x0_quad, unstable_eta)
ax2.plot(xs_div, color='purple', label=f'\u03b7 = {unstable_eta}')
ax2.set_title('Quadratic GD — Divergence  (\u03b7 = 2.1 > 2)')
ax2.set_xlabel('Iteration')
ax2.set_ylabel('x_t')
ax2.legend()

plt.tight_layout()
plt.savefig('fig_08_quadratic_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

For $q(x) = \tfrac{1}{2}x^2$, the tested stable step sizes $\eta \in \{0.05, 0.10, 0.15, 0.20\}$ all converge to $x^\star = 0$. At $\eta = 2.1$, the iterate diverges geometrically as $x_t = (-1.1)^t x_0$, which matches the threshold behavior above.

## 14. Final Commentary

**Takeaways.**
1. The finite-difference check confirms `df` on the smooth branches (max error below the runtime tolerance).
2. GD outcome depends on initialization: for the tested starts at $\eta = 0.05$, those below $x_{\text{max}} \approx 2.155$ converge to $x^\star$ and those above it oscillate around the nonsmooth local minimizer at the kink; sufficiently extreme starts can instead diverge (the guard exists for this reason).
3. Constant-step SGD settles into a noise floor, while a diminishing schedule keeps reducing the objective gap.

**Limitation.** The kink regime is studied numerically under a fixed right-branch subgradient convention; no theoretical treatment of subgradient dynamics at $x = 3$ is attempted.

**Bridge to Week 2.** Correct gradients, stable step sizes, and clear diagnostics precede model complexity — the same discipline carries directly into backpropagation.